In [1]:
# Cell: The Clinical Manifold (Z_clin)
import pandas as pd
import numpy as np
import umap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.preprocessing import StandardScaler
import os

print("[*] Initializing Clinical Manifold Construction...")

base_dir = "/workspace"
# Note: Adjust these filenames if your clinical tensors are named differently
clinical_file = os.path.join(base_dir, "data/processed/clinical_tensors/clinical_master_raw.csv.gz")

# 1. Load the Clinical Data
print("[*] Loading MIMIC-III Clinical Data...")
df_clin = pd.read_csv(clinical_file)

# Ensure mortality is an integer label
if 'Mortality' not in df_clin.columns:
    # Fallback in case your target column is named differently
    target_col = [c for c in df_clin.columns if 'mortality' in c.lower() or 'label' in c.lower()][0]
    df_clin.rename(columns={target_col: 'Mortality'}, inplace=True)

df_clin['Mortality'] = df_clin['Mortality'].astype(int)

# 2. Extract the Target Demographics from the Genomic Space
# (We need the exact count of Alive vs Dead from our 6 clean cohorts)
target_dead = (label_array == 1).sum()
target_alive = (label_array == 0).sum()
total_target = target_dead + target_alive

print(f"[*] Synthetic Alignment Target: {total_target} patients ({target_alive} Alive, {target_dead} Dead)")

# 3. Stratified Sampling from MIMIC-III
clin_dead = df_clin[df_clin['Mortality'] == 1].sample(n=target_dead, random_state=42)
clin_alive = df_clin[df_clin['Mortality'] == 0].sample(n=target_alive, random_state=42)
df_clin_matched = pd.concat([clin_dead, clin_alive]).sample(frac=1, random_state=42).reset_index(drop=True)

# 4. Feature Extraction and Scaling
# Drop identifiers and labels to get pure clinical physiology
ignore_cols = ['subject_id', 'hadm_id', 'icustay_id', 'Mortality', 'Patient_ID']
clinical_features = [c for c in df_clin_matched.columns if c not in ignore_cols]

print(f"[*] Extracting {len(clinical_features)} physiological features...")
X_clin_raw = df_clin_matched[clinical_features].astype(float).values
y_clin = df_clin_matched['Mortality'].values

# Standardize vital signs (Z-score)
scaler_clin = StandardScaler()
X_clin_scaled = scaler_clin.fit_transform(X_clin_raw)

# 5. UMAP Trajectory Mapping
# We use a high n_neighbors here to preserve the global "trajectory" of physical decline
print("[*] Running Trajectory Embedding on Clinical Vitals...")
reducer_clin = umap.UMAP(n_neighbors=50, min_dist=0.1, random_state=42)
Z_clin_2D = reducer_clin.fit_transform(X_clin_scaled)

# Save the high-dimensional space for the GW-OT alignment in the next step
# We use 15 dimensions to capture the full physiological variance without noise
pca_clin = PCA(n_components=15, random_state=42)
Z_clin_highD = pca_clin.fit_transform(X_clin_scaled)
print(f"[+] Extracted Z_clin shape for alignment: {Z_clin_highD.shape}")

# 6. Plot the Clinical Trajectory
plt.figure(figsize=(10, 8))
cmap_mortality = mcolors.ListedColormap(['#1f77b4', '#d62728']) 

scatter = plt.scatter(
    Z_clin_2D[:, 0], Z_clin_2D[:, 1],
    c=y_clin, cmap=cmap_mortality, s=20, alpha=0.8
)

plt.title("The Clinical Manifold ($Z_{clin}$)\n(MIMIC-III Physiological Trajectory)", fontsize=16)
plt.xticks([]); plt.yticks([])

from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', label='Alive (0)', markerfacecolor='#1f77b4', markersize=10),
                   Line2D([0], [0], marker='o', color='w', label='Dead (1)', markerfacecolor='#d62728', markersize=10)]
plt.legend(handles=legend_elements, loc='upper right', fontsize=12)

plt.tight_layout()
# --- The Magic Saving Lines ---
fig_dir = os.path.join(base_dir, "outputs/figures")
os.makedirs(fig_dir, exist_ok=True)
save_path = os.path.join(fig_dir, "YOUR_FILENAME_HERE.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"[*] Figure saved to: {save_path}")
# ------------------------------
plt.show()

[*] Initializing Clinical Manifold Construction...
[*] Loading MIMIC-III Clinical Data...


IndexError: list index out of range

In [2]:
import pandas as pd
import os
base_dir = "/workspace"
df_clin = pd.read_csv(os.path.join(base_dir, "data/processed/clinical_tensors/clinical_master_raw.csv.gz"), nrows=5)
print("Available Clinical Columns:")
print(df_clin.columns.tolist()[:30]) # Printing the first 30 columns

Available Clinical Columns:
['Patient_ID', 'Age', 'Gender', 'ICU_Length_of_Stay', 'Sepsis_Outcome', 'HR_mean', 'HR_min', 'HR_max', 'O2Sat_mean', 'O2Sat_min', 'O2Sat_max', 'Temp_mean', 'Temp_min', 'Temp_max', 'SBP_mean', 'SBP_min', 'SBP_max', 'MAP_mean', 'MAP_min', 'MAP_max', 'DBP_mean', 'DBP_min', 'DBP_max', 'Resp_mean', 'Resp_min', 'Resp_max', 'EtCO2_mean', 'EtCO2_min', 'EtCO2_max', 'BaseExcess_mean']
